In [48]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [49]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

df = pd.read_csv(url, names=columns)

print(df.shape)
df.head()

(768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [50]:
missing_cols = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

zero_counts = (df[missing_cols] == 0).sum()

print(zero_counts)

Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64


In [51]:
df[missing_cols] = df[missing_cols].replace(0, np.nan)

In [52]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [53]:
imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

In [54]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [55]:
dt_raw = DecisionTreeClassifier(random_state=42)

dt_raw.fit(X_train, y_train)

train_pred = dt_raw.predict(X_train)
test_pred = dt_raw.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy:", accuracy_score(y_test, test_pred))

print("Train F1:", f1_score(y_train, train_pred))
print("Test F1:", f1_score(y_test, test_pred))

Train Accuracy: 1.0
Test Accuracy: 0.6818181818181818
Train F1: 1.0
Test F1: 0.5148514851485149


In [56]:
dt_scaled = DecisionTreeClassifier(random_state=42)

dt_scaled.fit(X_train_scaled, y_train)

scaled_pred = dt_scaled.predict(X_test_scaled)

print("Scaled Test Accuracy:",
      accuracy_score(y_test, scaled_pred))

print("Scaled Test F1:",
      f1_score(y_test, scaled_pred))

Scaled Test Accuracy: 0.6818181818181818
Scaled Test F1: 0.5148514851485149


In [57]:
raw_pred = dt_raw.predict(X_test)

print(
    "Same predictions:",
    np.array_equal(raw_pred, scaled_pred)
)

Same predictions: True


In [58]:
dt_pruned = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=10,
    random_state=42
)

dt_pruned.fit(X_train, y_train)

pruned_pred = dt_pruned.predict(X_test)

print("Accuracy:",
      accuracy_score(y_test, pruned_pred))

print("F1:",
      f1_score(y_test, pruned_pred))

Accuracy: 0.7922077922077922
F1: 0.7090909090909091


In [59]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Accuracy:",
      accuracy_score(y_test, rf_pred))

print("F1:",
      f1_score(y_test, rf_pred))

Accuracy: 0.7662337662337663
F1: 0.64


In [60]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 4, 6, 8],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf_base = RandomForestClassifier(
    random_state=42
)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV F1:",
      grid_search.best_score_)

Best Parameters:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best CV F1: 0.653616645049816


In [61]:
best_rf = grid_search.best_estimator_

tuned_pred = best_rf.predict(X_test)

print("Test Accuracy:",
      accuracy_score(y_test, tuned_pred))

print("Test F1:",
      f1_score(y_test, tuned_pred))

Test Accuracy: 0.7402597402597403
Test F1: 0.5918367346938775


In [62]:
results = pd.DataFrame({
    "Model": [
        "Decision Tree - Unconstrained Raw",
        "Decision Tree - Unconstrained Scaled",
        "Decision Tree - Pre-Pruned",
        "Random Forest - Default",
        "Random Forest - Tuned"
    ],
    "Accuracy": [
        accuracy_score(y_test, raw_pred),
        accuracy_score(y_test, scaled_pred),
        accuracy_score(y_test, pruned_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, tuned_pred)
    ],
    "F1-score": [
        f1_score(y_test, raw_pred),
        f1_score(y_test, scaled_pred),
        f1_score(y_test, pruned_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, tuned_pred)
    ]
})

results

,Model,Accuracy,F1-score
0,Decision Tree - Unconstrained Raw,0.681818,0.514851
1,Decision Tree - Unconstrained Scaled,0.681818,0.514851
2,Decision Tree - Pre-Pruned,0.792208,0.709091
3,Random Forest - Default,0.766234,0.640000
4,Random Forest - Tuned,0.740260,0.591837


In [63]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_rf.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance

,Feature,Importance
1,Glucose,0.267931
5,BMI,0.159848
6,DiabetesPedigreeFunction,0.124381
7,Age,0.121357
4,Insulin,0.090604
2,BloodPressure,0.084977
0,Pregnancies,0.079366
3,SkinThickness,0.071537


In [64]:
importance["Rank"] = range(1, len(importance) + 1)

importance

,Feature,Importance,Rank
1,Glucose,0.267931,1
5,BMI,0.159848,2
6,DiabetesPedigreeFunction,0.124381,3
7,Age,0.121357,4
4,Insulin,0.090604,5
2,BloodPressure,0.084977,6
0,Pregnancies,0.079366,7
3,SkinThickness,0.071537,8
